---
title: Functional Programming
abstract: |
    In this notebook, we explore key techniques in functional programming, with a particular emphasis on recursion. Recursion exemplifies code reuse by defining a function in terms of itself, allowing for elegant, declarative solutions to complex problems using a divide-and-conquer approach. However, readers will also learn about the potential inefficiencies of recursion due to redundant computations of subproblem solutions. To address these inefficiencies, we introduce the concept of using function states to simplify calculations, while discussing the pitfalls of global variables and their impact on program predictability. Finally, we delve into the idea of encapsulation through closures, paving the way for an understanding of object-oriented programming principles.
---

## Recursion

To motivate the idea of programming using functions, consider the problem of computing the greatest common divisor:

::::{prf:definition} gcd
:label: def:gcd

The greatest common divisor $\operatorname{gcd}(a, b)$ of two non-zero integers $a$ and $b$ is the largest integer $d$ that divides both $a$ and $b$, i.e., $d|a$ and $d|b$.

::::

::::{exercise}
:label: ex:gcd 

Implement the function `gcd(a, b)` to return the greatest common divisor of the non-zero integer `a` and `b`.

::::

In [ ]:
def gcd(a, b):
    # YOUR CODE HERE
    raise NotImplementedError


print(gcd(3 * 5, 5 * 7))

In [ ]:
# tests
a, b = 3*5, 5*7
assert gcd(a, b) == gcd(a, -b) == gcd(-a, b) == 5
assert gcd(10**10, 10**10 + 1) == 1

One may implement [](#def:gcd) directly using a for loop as follows:

In [ ]:
def gcd(a, b):
    if a and b:
        for d in range(min(abs(a), abs(b)), 0, -1):
            if a % d == b % d == 0:
                return d
    return abs(a) or abs(b)


print(gcd(3 * 5, 5 * 7))

Unfortunately, the implementation is inefficient. It will take a long time to run the following test case:

In [ ]:
if input('Run? [Y/n]').lower() != 'n':
    assert gcd(10**10, 10**10 + 1) == 1

A more efficient way to compute gcd is as follows:

::::{prf:proposition} [Euclidean algorithm for gcd](https://en.wikipedia.org/wiki/Euclidean_algorithm)
:label: pro:gcd 

The gcd in [](#def:gcd) satisfies the following recurrence relation

$$
\operatorname{gcd}(a,b)
= \operatorname{gcd}(b, a\operatorname{mod}b)
$$ (eq:gcd)

except for the base case $\operatorname{gcd}(a, 0)=\lvert a\rvert$.[^gcd]

::::

[^gcd]: What about the other base case $\operatorname{gcd}(0, b)$?

::::{prf:proof}
:class: dropdown
:nonumber:

The base case holds trivially and $\operatorname{gcd}(a, b)=\operatorname{gcd}(b, a)$, so it suffices to show that any common factor of $a$ and $b$ must divide $r:=a\operatorname{mod}b$, if $b\neq 0$. To this end, suppose $c$ is the common factor. Then, for some integers $q_0, q_1, q_2$:

\begin{align}
r &\stackrel{\text{(a)}}= \underbrace{a}_{\stackrel{\text{(b)}}=q_1 c} - \underbrace{b}_{\stackrel{\text{(c)}}=q_2 c} q_0\\
&= (q_1-q_2 q_0) c
\end{align}

which is divisible by $c$ as desired. (Can you explain the equalities $\text{(a)--(c)}$?)

::::

How to implement [](#eq:gcd)? Can we define `gcd` using `gcd`?

::::{exercise}
:label: ex:base-case

The following is a nearly correct implementation of [](#def:gcd) except for a common mistake that often results in an infinite loop/recursion.

::::

In [ ]:
def gcd(a, b):
    return gcd(a % b, b)


print(gcd(3 * 5, 5 * 7))

YOUR ANSWER HERE

::::{seealso} Recursion

A function that calls itself (*recurs*) is known as a
[*recursion*](https://en.wikipedia.org/wiki/Recursion_(computer_science)). In
Python, as well as in most of the modern programming languages, it is perfectly
legitimate to define such a function. This technique allows us to reuse the
code within the function to define the function itself, taking code reuse to
the extreme!

- Recursion is often shorter and easier to understand. It can provide *elegant*
solutions to complex problems.
- Recursion can be easier to write code by *wishful thinking* or *[declarative
programming](https://en.wikipedia.org/wiki/Declarative_programming)* as opposed
to [imperative
programming](https://en.wikipedia.org/wiki/Imperative_programming).

::::

In [ ]:
%%hermes --no-context
Using Euclidean algorithm for gcd, explain in one paragraph or two how to come 
up with a recursion that solves a problem by divide-and-conquer.

::::{seealso}
:class: dropdown

**Coming up with divide-and-conquer recursion** follows two key steps. First, find an **inductive reduction**: show that solving the problem on a smaller instance gives you the answer to the original problem. In the Euclidean algorithm, the critical insight is `gcd(a, b) = gcd(b, a mod b)` — the GCD of two numbers equals the GCD of the smaller number and the remainder. This reduces a large problem to a strictly smaller one (since `a mod b < b`), guaranteeing progress toward a base case. Second, identify the **base case** where the problem is trivially solvable: `gcd(a, 0) = a`. Once you have the reduction rule and the base case, you just write them down directly as the recursive step and the termination condition.

The general recipe for any divide-and-conquer recursion is the same: (1) ask "can I express this problem in terms of a smaller subproblem of the same kind?" — that's your recursive step; and (2) ask "when is the problem small enough to solve immediately?" — that's your base case. The Euclidean algorithm works because the modular reduction shrinks the input monotonically, so the recursion is guaranteed to terminate.

::::

## Recursion vs Iteration

**Is recursion always better than iteration?**

Consider computing the [Fibonacci
number](https://en.wikipedia.org/wiki/Fibonacci_number) of order $n$, which is
defined in a recursive manner by the following [recurrence
relation](https://en.wikipedia.org/wiki/Recurrence_relation):[^fib]

$$
F_n := 
\begin{cases}
F_{n-1}+F_{n-2} & n>1 \kern1em \text{(recurrence)}\\
1 & n=1 \kern1em \text{(base case)}\\
0 & n=0 \kern1em \text{(base case)}.
\end{cases}
$$

[^fib]: Fibonacci numbers have practical applications in generating
[pseudorandom
numbers](https://en.wikipedia.org/wiki/Lagged_Fibonacci_generator).

In [ ]:
%%hermes --no-context
Describe what the fibonacci number is in one sentence and list three most
important applications of the Fibonacci number in bullet points.

::::{seealso}
:class: dropdown

**Fibonacci numbers** are a sequence where each number is the sum of the two preceding ones, starting from 0 and 1: 0, 1, 1, 2, 3, 5, 8, 13, 21, …

**Three important applications:**

- **Algorithm analysis & complexity** — Fibonacci numbers appear in the worst-case analysis of algorithms like Fibonacci heaps, divide-and-conquer recurrences, and the worst-case input for naive recursive algorithms (making it the canonical example for teaching recursion vs. memoization).
- **Natural phenomena & modeling** — They describe growth patterns found throughout biology: phyllotaxis (leaf arrangement), branching trees, branching in coral reefs, and the spirals in sunflower seeds and pinecones.
- **Financial analysis & trading** — Fibonacci retracement and extension levels are widely used in technical analysis to predict potential support and resistance price levels in markets.
::::

The following function `fibonacci(n)` implements $F_n$ naturally as a
recursion:

In [ ]:
%%pytutor
def fibonacci(n):
    if n > 1:
        return fibonacci(n - 1) + fibonacci(n - 2)  # recursion
    elif n == 1:
        return 1
    else:
        return 0


print(fibonacci(2))

::::{exercise}
:label: ex:fibonacci_effficiency

Find the smallest value of `n` for `fibonacci(n)` to run for more than a second.

:::{hint}
:class: dropdown

Simply bind `n` to an appropriate value by observing the running time reported by the `timeit` magic. You can run `%timeit?` to see the docstring.
:::

::::

In [ ]:
# Assign n the appropriate value
# YOUR CODE HERE
raise NotImplementedError
n

In [ ]:
%%timeit -n 1 -r 1
fibonacci(n)

In [ ]:
# hidden tests

**Is the recursion efficient?**

As a comparison, the following computes the Fibonacci number using a while loop instead of a recursion.

In [ ]:
%%pytutor
def fibonacci_iteration(n):
    if n > 1:
        _, F = 0, 1  # next two Fibonacci numbers
        while n > 1:
            _, F, n = F, F + _, n - 1
        return F
    elif n == 1:
        return 1
    else:
        return 0


fibonacci_iteration(3)

::::{exercise}
:label: ex:fibonacci_iteration_efficiency

Find the smallest values of `n` for `fibonacci_iteration(n)` to run for more than a second.

::::

In [ ]:
# Assign n the appropriate value
# YOUR CODE HERE
raise NotImplementedError
n

In [ ]:
%%timeit -n 1 -r 1
fibonacci_iteration(n)

In [ ]:
# hidden tests

To understand the difference in performance, modify `fibonacci` to print each function call as follows.

In [ ]:
def fibonacci_verbose(n):
    """Returns the Fibonacci number of order n."""
    print(f"fibonacci({n})")
    return fibonacci_verbose(n - 1) + fibonacci_verbose(n - 2) if n > 1 else 1 if n == 1 else 0


fibonacci_verbose(5)

::::{exercise}
:label: ex:fib_recursion_iteration

Why `fibonacci(n)` is much slower than `fibonacci_iteration(n)`?

::::

YOUR ANSWER HERE

**Setting performance considerations aside, do we really need recursion?**

Indeed, we can always convert a recursion to an iteration that is at least as efficient. (Why?)[^step]

[^step]: A recursion is ultimately executed step-by-step with an execution stack that keeps track of the recursive calls. Such an execution essentially converts a recursion to an iteration.

::::{exercise}
:label: ex:gcd_iteration 

Complete the following function `gcd_iteration` to re-implement [](#pro:gcd)
using a while loop instead of recursion.

:::{hint}
:class: dropdown
See [tail
recursion](https://en.wikipedia.org/wiki/Recursion_(computer_science)#Tail-recursive_functions).
:::

::::

In [ ]:
%%pytutor
def gcd_iteration(a, b):
    # YOUR CODE HERE
    raise NotImplementedError


gcd_iteration(3 * 5, 5 * 7)

In [ ]:
%%hermes --no-context
Explain in one paragraph what a tail recursion is and how to convert a tail
recursion to an iteration.

::::{seealso}
:class: dropdown

Tail recursion occurs when a recursive call is the very last operation in a function — no further computation happens after the call returns, meaning the current stack frame holds nothing more to do and can be discarded. Because of this, a tail-recursive function can be converted directly into an iteration by replacing the function’s parameters with local variables and wrapping the body in a loop that updates those variables each iteration instead of making a recursive call. The key pattern is: (1) initialize the loop variables from the original arguments (often including an accumulator that carries the intermediate result), (2) repeat the body’s computation in a while loop, updating the variables to what would have been the next recursive call’s arguments, and (3) return the accumulator when the base case is reached. This transformation preserves the logic while eliminating the call-stack growth that ordinary recursion incurs.
::::

In [ ]:
%%hermes --no-context
Explain in one paragraph how non-tail recursive functions can be significantly
slower than their iterative counterparts.

::::{seealso}
:class: dropdown

Non-tail recursive functions can be significantly slower than their iterative counterparts because each recursive call pushes a new stack frame onto the call stack — a bookkeeping operation that saves local variables, return addresses, and registers before jumping to the next call. For a function that recurses N times, this means N stack allocations and N deallocations on the way back, each incurring memory access overhead that an iterative loop completely avoids by reusing the same stack frame throughout. Additionally, the deep chain of nested calls can exhaust the stack, causing stack overflow, while also evicting instruction-cache entries and preventing the CPU branch predictor from settling into a stable pattern — whereas a tight loop runs in-place with minimal overhead, excellent cache locality, and predictable branching.

::::

## Global Variables

Suppose our task is to print the entire sequence of Fibonacci numbers up to certain order such as:

In [ ]:
for n in range(10):
    print(fibonacci_iteration(n))

::::{exercise}
:label: ex:efficient 

The above loop is inefficient. Why?

::::

YOUR ANSWER HERE

**How to avoid redundant computations?**

One way is to store the last two computed Fibonacci numbers as *global* variables.

In [ ]:
%%pytutor
Fn, Fnn, n = 0, 1, 0  # global variables


def print_fibonacci_state():
    print(
        f"""Global states:
    Fn  : Next Fibonacci number      = {Fn}
    Fnn : Next next Fibonacci number = {Fnn}
    n   : Next order                 = {n}"""
    )


def next_fibonacci():
    global Fn, Fnn, n  # global declaration
    value, Fn, Fnn, n = Fn, Fnn, Fn + Fnn, n + 1
    return value


for i in range(5):
    print(next_fibonacci())
print_fibonacci_state()

::::{seealso} Rules for [*global/local variables*](https://docs.python.org/3/faq/programming.html#what-are-the-rules-for-local-and-global-variables-in-python)

1. A local variable must be defined within a function.
1. An assignment defines a local variable except after a [`global` statement](https://docs.python.org/3/reference/simple_stmts.html#the-global-statement).
::::

::::{note} **Why `global` is NOT needed in `print_fibonacci_state`?**
:class: dropdown

Without ambiguity, `Fn, Fnn, n` in `print_fibonacci_state` are not local variables by Rule 1 because they are not defined within the function.

::::

**Why `global` is needed in `next_fibonacci`?**

What would happen if the `global` statement is removed?

In [ ]:
%%pytutor
def next_fibonacci():
    """Returns the next Fibonacci number."""
    # global Fn, Fnn, n
    value = n
    Fn, Fnn, n = Fnn, Fn + Fnn, n + 1
    return value


next_fibonacci()

`UnboundLocalError` is raised (as opposed to `NameError`) because

- the assignment in Line 5 defines `n` as a local variable by Rule 2, but
- the assignment in Line 4 references `n`, which is not yet defined at that point.

Consider rewriting the for loop as a while loop:

In [ ]:
%%pytutor
Fn, Fnn, n = 0, 1, 0  # global variables


def print_fibonacci_state():
    print(
        f"""Global states:
    Fn  : Next Fibonacci number      = {Fn}
    Fnn : Next next Fibonacci number = {Fnn}
    n   : Next order                 = {n}"""
    )


def next_fibonacci():
    """Returns the next Fibonacci number."""
    global Fn, Fnn, n  # global declaration
    value, Fn, Fnn, n = Fn, Fnn, Fn + Fnn, n + 1
    return value


n = 0
while n < 5:
    print(next_fibonacci())
    n += 1
print_fibonacci_state()

::::{exercise}
:label: ex:bug_fib 

Why does the while loop print only 3 numbers instead of 5 Fibonacci numbers?

::::

YOUR ANSWER HERE

To avoid such error, a convention in Python is to use a leading underscore for variable names that are [*private*](https://www.python.org/dev/peps/pep-0008) (for internal use):  
> _single_leading_underscore: weak "internal use" indicator. E.g., from M import * does not import objects whose names start with an underscore.

In [ ]:
%%pytutor
_Fn, _Fnn, _n = 0, 1, 0  # global variables


def print_fibonacci_state():
    print(
        f"""Global states:
    _Fn  : Next Fibonacci number      = {_Fn}
    _Fnn : Next next Fibonacci number = {_Fnn}
    _n   : Next order                 = {_n}"""
    )


def next_fibonacci():
    """Returns the next Fibonacci number."""
    global _Fn, _Fnn, _n  # global declaration
    value, _Fn, _Fnn, _n = _Fn, _Fnn, _Fn + _Fnn, _n + 1
    return value


n = 0
while n < 5:
    print(next_fibonacci())
    n += 1
print_fibonacci_state()

::::{important} What is wrong with global variables?

Using global variables,
- codes are less predictable, more difficult to reuse/extend, and
- tests cannot be isolated, making debugging difficult.
::::

## Closure

**Is it possible to store the function states without using global variables?**

We can use nested functions and [`nonlocal` variables](https://docs.python.org/3/reference/simple_stmts.html#the-nonlocal-statement).

In [ ]:
def create_fibonacci(Fn, Fnn):
    def next_fibonacci():
        """Returns the next (generalized) Fibonacci number starting with
        Fn and Fnn as the first two numbers."""
        nonlocal Fn, Fnn, n  # declare nonlocal variables
        value = Fn
        Fn, Fnn, n = Fnn, Fn + Fnn, n + 1
        return value

    def print_fibonacci_state():
        print(
            """States:
        Next Fibonacci number      = {}
        Next next Fibonacci number = {}
        Next order                 = {}""".format(
                Fn, Fnn, n
            )
        )

    n = 0  # Fn and Fnn specified in the function arguments
    return next_fibonacci, print_fibonacci_state


next_fibonacci, print_fibonacci_state = create_fibonacci(0, 1)
n = 0
while n < 5:
    print(next_fibonacci())
    n += 1
print_fibonacci_state()

The state variables `Fn`, `Fnn`, and `n` are now [*encapsulated*](https://en.wikipedia.org/wiki/Encapsulation_(computer_programming)), meaning they are contained within the *scope* of the `create_fibonacci` function, i.e., 
- they are not exposed globally, unlike the global variables, but
- they are accessible by the inner functions of `create_fibonacci`.

The encapsulation allows us to create multiple Fibonacci sequences with different base cases independently without interfering with each others:

In [ ]:
usual_fib = create_fibonacci(0, 1)
cs1302_fib = create_fibonacci("cs", "1302")
for n in range(3):
    print(usual_fib[0]())
    usual_fib[1]()
    print(cs1302_fib[0]())
    cs1302_fib[1]()

`next_fibonacci` and `print_fibonacci_state` are *local functions* of `create_fibonacci`:

- Local functions can access (*capture*) the other local variables of `create_fibonacci` by forming the so-called *closures*. Each local function has an attribute named `__closure__` that stores the captured local variables.
- Similar to the `global` statement, a [`nonlocal` statement](https://docs.python.org/3/reference/simple_stmts.html#the-nonlocal-statement) is needed for assigning non-local variables.

In [ ]:
def print_closure(f):
    """Print the closure of a function."""
    print("closure of ", f.__name__)
    for cell in f.__closure__:
        print("    {} content: {!r}".format(cell, cell.cell_contents))


print_closure(next_fibonacci)
print_closure(print_fibonacci_state)

::::{seealso} Lexical/static scoping

[Lexical scoping](https://en.wikipedia.org/wiki/Scope_(computer_science)#Lexical_scope) is a key concept that allows local functions to capture and remember the scope in which they were created. When a local function is defined inside another function, it forms a closure that retains access to the variables in the outer function’s scope, even after the outer function has finished executing. This is possible because the function's `__closure__` attribute stores references to these variables. By using the `nonlocal` statement, you can modify these captured variables from within the local function, providing a flexible and powerful way to manage state and behavior in your programs.

::::

Lexical scoping can sometimes be counter-intuitive:

In [ ]:
def foo():
    return x


def bar():
    x = "dynamically scoped"
    # NOT the same as:
    # return x
    return foo()  


x = "lexically scoped"
bar()

Try re-executing the above code after uncommenting the line `# return x`. Even
though `foo()` returns `x`, the line `return foo()` is not the same as `return
x`, i.e., one cannot simply substitute `foo()` by `x`.

In [ ]:
%%hermes --no-context
Explain in one paragraph the differences between lexical scoping and dynamic
scoping, and why Python implements the prior.

::::{seealso}
:class: dropdown

Lexical scoping resolves a variable name by looking up the chain of textually enclosing blocks — the binding is determined by where the code is written and is fixed at compile time. Dynamic scoping resolves names by walking the call stack at runtime — a free variable looks through the caller’s local scope, then the caller’s caller, making the same identifier depend on who invoked the function rather than where it was defined. Python implements lexical scoping because it makes code behavior predictable: you can determine a name’s binding just by reading the source layout without tracing every possible call path, which enables static analysis, reliable IDE tooling, clear data-flow reasoning, and avoids the subtle bugs that arise when unrelated functions unintentionally share caller-level state.

::::

Lexical scoping is a powerful concept that can be leveraged to implement
object-oriented programming. For example, we can rewrite `create_fibonacci` to
return a Fibonacci object as shown below:

In [ ]:
def create_fibonacci(Fn, Fnn):
    def next():
        """Returns the next (generalized) Fibonacci number starting with
        Fn and Fnn as the first two numbers."""
        nonlocal Fn, Fnn, n
        value = Fn
        Fn, Fnn, n = Fnn, Fn + Fnn, n + 1
        return value

    def self():  # make the return object callable to replace
        print(
            """States:
        Next Fibonacci number      = {}
        Next next Fibonacci number = {}
        Next order                 = {}""".format(
                Fn, Fnn, n
            )
        )

    n = 0

    self.next = next  # add next as an attribute of self
    return self       # to be returned


fib = create_fibonacci(0, 1)
n = 0
while n < 5:
    print(fib.next())
    n += 1
fib()

The `create_fibonacci` function returns the `self` function, which has access
to the `next` function and other internal states of the `create_fibonacci`
function. To create multiple Fibonacci objects:

In [ ]:
usual_fib = create_fibonacci(0, 1)
cs1302_fib = create_fibonacci("cs", "1302")
for n in range(3):
    print(usual_fib.next())
    usual_fib()
    print(cs1302_fib.next())
    cs1302_fib()

As the above code shows, closures enable an object-oriented programming
approach by allowing the creation of objects that share methods but maintain
possibly distinct attribute values.

In [ ]:
%%hermes --no-context
Explain in one paragraph the importance of closures in programming and why they
are called closures.

A closure is a function that captures and retains access to the variables in its surrounding scope even after that scope has finished executing. They are called “closures” because the function closes over — binds to and carries with it — the environment in which it was created, preserving those external bindings rather than letting them go out of scope. Closures are important because they enable functions to carry private state without global variables, support powerful patterns like callbacks, partial application, decorators, and event handlers, and provide a clean bridge between data and behavior: a closure bundles a computation with the data it needs into a single, first-class value that can be passed around, stored, and invoked later, all while keeping that data encapsulated and inaccessible from the outside.

:::{exercise} agentic functional transformation
:label: ex:map-filter-agentic

Ask an AI agent to rewrite a `for` loop with a `map()` or
`filter()` expression. Evaluate whether the functional version
is actually more readable for your specific case.

::::

YOUR ANSWER HERE

In [ ]:
%%hermes --no-context
How to define static variables in Python?

::::{seealso}
:class: dropdown

Python doesn't have a `static` keyword for local variables, but you can achieve the same effect — a variable that persists across calls while staying local to one function — using a few common approaches:

**1. Function attributes (most common and explicit)**

```python
def counter():
    counter.n += 1
    return counter.n

counter.n = 0  # initialise
```

**2. Mutable default argument**

```python
def counter(n=[0]):
    n[0] += 1
    return n[0]
```

This works because the default list object is created once at function definition time and reused on every call. It's a well-known Python idiom but can look like a bug to those unfamiliar with it.

**3. Closure with a nonlocal variable**

```python
def make_counter():
    n = 0
    def counter():
        nonlocal n
        n += 1
        return n
    return counter

counter = make_counter()
```

**4. Using a class**

```python
class Counter:
    n = 0
    @classmethod
    def count(cls):
        cls.n += 1
        return cls.n
```

The **function attribute** approach is generally preferred because it's the most Pythonic, readable, and keeps everything attached to the function itself without workarounds.

::::

In [ ]:
%%hermes --no-context
Why is closure as a thread-safe alternative to static variables?

::::{seealso}
:class: dropdown

A closure isn’t thread-safe because it has built-in locking — it’s thread-safe because it eliminates shared state entirely. With a static variable, every thread calling the same function shares one piece of mutable data, requiring locks or atomics to avoid race conditions. With a closure, you create a factory function that returns a new closure for each consumer — each closure captures its own private copy of the enclosed variables. If every thread gets its own closure instance, there is no shared mutable state to contend over, so no synchronization is needed. This is the core principle of functional programming: state is isolated inside independent objects rather than pooled globally, making the program inherently free of data races without any locking overhead.

::::

:::{exercise} agentic reduce pattern
:label: ex:reduce-agentic

Ask an AI agent to use `functools.reduce()` to find the
maximum element without calling `max()`. Discuss why
`max()` is the better choice in practice.

::::

YOUR ANSWER HERE